# DL_04 연습문제 - 신경망 학습 (손실함수, 미니배치)

MSE, CEE, 배치용 CEE, 미니배치 추출, 그리고 개념 문제까지.

### Q1. MSE & CEE 직접 계산
정답 t = [0,0,1,0,0,0,0,0,0,0] (one-hot, 정답=2번)

1. y1 = [0.1,0.05,0.6,0,0.05,0.1,0,0.1,0,0]   ← 정답을 잘 맞춤
2. y2 = [0.1,0.05,0.1,0,0.05,0.1,0,0.6,0,0]   ← 정답을 못 맞춤

각각 MSE와 CEE를 구하고, **둘 중 어느 쪽이 더 나은 예측**인지 손실 값으로 판정하세요.

In [ ]:
import numpy as np

def mse(y, t):
    # TODO
    pass

def cee(y, t, delta=1e-7):
    # TODO
    pass

t  = np.array([0,0,1,0,0,0,0,0,0,0])
y1 = np.array([0.1,0.05,0.6,0,0.05,0.1,0,0.1,0,0])
y2 = np.array([0.1,0.05,0.1,0,0.05,0.1,0,0.6,0,0])

In [ ]:
def mse(y, t):
    return 0.5 * np.sum((y - t)**2)

def cee(y, t, delta=1e-7):
    return -np.sum(t * np.log(y + delta))

t  = np.array([0,0,1,0,0,0,0,0,0,0])
y1 = np.array([0.1,0.05,0.6,0,0.05,0.1,0,0.1,0,0])
y2 = np.array([0.1,0.05,0.1,0,0.05,0.1,0,0.6,0,0])

print('y1: MSE=%.4f CEE=%.4f' % (mse(y1,t), cee(y1,t)))
print('y2: MSE=%.4f CEE=%.4f' % (mse(y2,t), cee(y2,t)))
# y1의 손실이 더 작음 -> y1이 더 나은 예측

### Q2. CEE에서 delta(1e-7) 를 더하는 이유
한 줄로 답하세요.

<details><summary>정답</summary>

`np.log(0)` = `-inf` 가 되어 이후 계산이 모두 `nan/inf` 로 오염되는 것을 방지하기 위해 아주 작은 값을 더해줌.
</details>

### Q3. 손실 함수 수식 쓰기
1. MSE 수식을 쓰세요.
2. CEE 수식을 쓰세요.
3. 미니배치 CEE (배치 크기 N) 수식을 쓰세요.

<details><summary>정답</summary>

$$E_{MSE} = \frac{1}{2}\sum_{k}(y_k - t_k)^2$$
$$E_{CEE} = -\sum_{k} t_k \log y_k$$
$$E_{batch} = -\frac{1}{N}\sum_{n}\sum_{k} t_{nk} \log y_{nk}$$
</details>

### Q4. 배치용 CEE 구현 (★ 시험 출제)
두 가지 버전을 구현하세요.

1. **t가 원-핫 인코딩** 일 때: `cross_entropy_error1(y, t)`
2. **t가 정답 레이블 인덱스** 일 때 (예: t=[2,7,0,...]): `cross_entropy_error2(y, t)`

공통 조건
- y가 1차원이면 (1,N) 으로 reshape 후 처리
- 배치 크기로 나눠서 평균 손실 반환

In [ ]:
def cross_entropy_error1(y, t):
    # 원-핫 버전
    # TODO
    pass

def cross_entropy_error2(y, t):
    # 레이블 인덱스 버전
    # TODO
    pass

# 테스트
y = np.array([[0.1,0.05,0.6,0,0.05,0.1,0,0.1,0,0],
              [0.1,0.05,0.1,0,0.05,0.1,0,0.6,0,0]])
t_oh    = np.array([[0,0,1,0,0,0,0,0,0,0],
                    [0,0,0,0,0,0,0,1,0,0]])
t_label = np.array([2, 7])

# print(cross_entropy_error1(y, t_oh))
# print(cross_entropy_error2(y, t_label))

In [ ]:
def cross_entropy_error1(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size

def cross_entropy_error2(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
    batch_size = y.shape[0]
    # y[행 인덱스, 정답열 인덱스] 만 뽑아서 log
    return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size

y = np.array([[0.1,0.05,0.6,0,0.05,0.1,0,0.1,0,0],
              [0.1,0.05,0.1,0,0.05,0.1,0,0.6,0,0]])
t_oh    = np.array([[0,0,1,0,0,0,0,0,0,0],
                    [0,0,0,0,0,0,0,1,0,0]])
t_label = np.array([2, 7])

print('one-hot CEE :', cross_entropy_error1(y, t_oh))
print('label   CEE :', cross_entropy_error2(y, t_label))   # 동일해야 함

### Q5. 미니배치 무작위 추출
훈련 데이터가 60000장 있다고 가정. 그 중 10장을 무작위로 골라 미니배치 인덱스를 만드는 코드를 작성하세요.

<details><summary>힌트</summary>

`np.random.choice(전체크기, 뽑을개수, replace=False)`
</details>

In [ ]:
train_size = 60000
batch_size = 10

# TODO: train_size 중 batch_size개 무작위 인덱스 뽑기
# batch_idx = ...
# print(batch_idx)

In [ ]:
batch_idx = np.random.choice(train_size, batch_size, replace=False)
print(batch_idx)

### Q6. 종합 개념 문제
1. **훈련 데이터와 시험 데이터** 를 나누는 이유는?
2. **과적합(overfitting)** 이란? 한 줄로.
3. 왜 **모든 데이터** 가 아니라 **미니배치** 로 학습하는가? 두 가지 이상 이유.
4. 정확도(accuracy)가 아니라 **손실 함수**를 학습 지표로 사용하는 이유는? (※ 시험 단골)

<details><summary>정답</summary>

1. 학습에 사용한 데이터가 아닌 **새 데이터**에서도 잘 동작하는지 = **일반화 성능** 을 측정하기 위해.
2. 훈련 데이터에는 잘 맞지만 새로운 데이터에는 잘 맞지 않는 상태.
3. (a) 전체 데이터로 한 번 갱신은 너무 느리고 메모리도 큼, (b) 일부 데이터의 손실로 전체 손실을 근사할 수 있음, (c) 노이즈가 들어가 지역 최소에 빠지는 것을 어느 정도 회피.
4. 정확도는 **계단형(불연속)** 이라 가중치를 살짝 바꿔도 거의 변하지 않거나 갑자기 튀어 미분이 0/정의 안됨 → 경사하강이 불가. 손실함수는 **연속이고 미분 가능** 하므로 기울기를 구해 가중치를 갱신할 수 있음.
</details>